# Build EMOPAIR Dataset

Using the results from `validate_LLMs.ipynb`, we can now build the EMOPAIR dataset.

Based on the Alt-Test results, we will use the following LLMs as annotators:
- Valence: No LLMs passed the Alt-Test for valence, so we will not use any LLMs as annotators for this dimension.
- Arousal: GPT-5.1
- Dominance: claude-sonnet-4-5

For Valence, we will use the prior EMOBANK data. 

In [ ]:
import os
import pandas as pd 
import pairadigm as pdm
import random 

import dotenv

dotenv.load_dotenv(dotenv_path='/fs/clip-projects/clip-k12/mlchrzan/.env')

## Prep

In [ ]:
# Load EMOBANK data to get the original text data
emobank = pd.read_csv('Data/emobank.csv')
emobank

In [ ]:
# Load CGCOT Prompts
with open('Data/cgcot_prompts/valence.txt', 'r') as file:
    val_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/arousal.txt', 'r') as file:
    aro_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/dominance.txt', 'r') as file:
    dom_prompts = file.read().splitlines()

## Build Pairs for pairadigms

In [ ]:
# NOTE: There are issues with the seeds where this does not guarantee the same pairs across runs. So here we will just load the pairs from a csv file that was generated with a fixed seed.
all_pairs = pd.read_csv('emopair_pairs.csv')

### Within split pairs

In [ ]:
train_pairs = pdm.pair_items(emobank.loc[emobank['split'] == 'train', 'id'].reset_index(drop=True), num_pairs_per_item=10)
dev_pairs = pdm.pair_items(emobank.loc[emobank['split'] == 'dev', 'id'].reset_index(drop=True), num_pairs_per_item=15)
test_pairs = pdm.pair_items(emobank.loc[emobank['split'] == 'test', 'id'].reset_index(drop=True), num_pairs_per_item=15)
print(f"Number of training pairs: {len(train_pairs)}")
print(f"Number of validation pairs: {len(dev_pairs)}")
print(f"Number of test pairs: {len(test_pairs)}")

### Between split pairs

In [ ]:
train_items = train_pairs['item1'].unique()
dev_items = dev_pairs['item1'].unique()
test_items = test_pairs['item1'].unique()

# Generate mixed pairs where item1 and item2 come from different splits
mixed_pairs = pd.DataFrame(columns=['item1', 'item2', 'item1_split', 'item2_split'])

# For reproducibility
random.seed(42)

# Helper to pair each item with n items from a specific split
def add_unique_mixed_pairs_per_split(item1_list, item1_split, target_items, target_split, n_pairs, mixed_pairs):
    """
    Pair each item in item1_list with n_pairs items from target_items (target_split).
    """
    for item1 in item1_list:
        count = 0
        attempts = 0
        
        while count < n_pairs and attempts < 100*n_pairs:
            item2 = random.choice(list(target_items))
            
            # Check for duplicates
            if mixed_pairs[(mixed_pairs['item1'] == item1) & (mixed_pairs['item2'] == item2)].empty:
                mixed_pairs = pd.concat([
                    mixed_pairs,
                    pd.DataFrame([{
                        'item1': item1,
                        'item2': item2,
                        'item1_split': item1_split,
                        'item2_split': target_split
                    }])
                ], ignore_index=True)
                count += 1
            attempts += 1
    
    return mixed_pairs

# Number of pairs per split
n_pairs_per_split = 2

# Train items: pair with dev and test
mixed_pairs = add_unique_mixed_pairs_per_split(train_items, 'train', dev_items, 'dev', n_pairs_per_split, mixed_pairs)
mixed_pairs = add_unique_mixed_pairs_per_split(train_items, 'train', test_items, 'test', n_pairs_per_split, mixed_pairs)

# Dev items: pair with train and test
mixed_pairs = add_unique_mixed_pairs_per_split(dev_items, 'dev', train_items, 'train', n_pairs_per_split, mixed_pairs)
mixed_pairs = add_unique_mixed_pairs_per_split(dev_items, 'dev', test_items, 'test', n_pairs_per_split, mixed_pairs)

# Test items: pair with train and dev
mixed_pairs = add_unique_mixed_pairs_per_split(test_items, 'test', train_items, 'train', n_pairs_per_split, mixed_pairs)
mixed_pairs = add_unique_mixed_pairs_per_split(test_items, 'test', dev_items, 'dev', n_pairs_per_split, mixed_pairs)

mixed_pairs

### Combine all pairs

In [ ]:
train_pairs['item1_split'] = 'train'
dev_pairs['item1_split'] = 'dev'
test_pairs['item1_split'] = 'test'
train_pairs['item2_split'] = 'train'
dev_pairs['item2_split'] = 'dev'
test_pairs['item2_split'] = 'test'

all_pairs = pd.concat([train_pairs, dev_pairs, test_pairs, mixed_pairs], ignore_index=True)
all_pairs

In [ ]:
# Join the text data from emobank for item1 and item2
all_pairs = all_pairs.merge(emobank[['id', 'text']], left_on='item1', right_on='id', how='left').rename(columns={'text': 'item1_text'}).drop(columns=['id'])
all_pairs = all_pairs.merge(emobank[['id', 'text']], left_on='item2', right_on='id', how='left').rename(columns={'text': 'item2_text'}).drop(columns=['id'])
all_pairs

In [ ]:
# Join the V, A, D ratings from emobank for item1 and item2
all_pairs = all_pairs.merge(emobank[['id', 'V']], left_on='item1', right_on='id', how='left').rename(columns={'V': 'item1_V'}).drop(columns=['id'])
all_pairs = all_pairs.merge(emobank[['id', 'A']], left_on='item1', right_on='id', how='left').rename(columns={'A': 'item1_A'}).drop(columns=['id'])
all_pairs = all_pairs.merge(emobank[['id', 'D']], left_on='item1', right_on='id', how='left').rename(columns={'D': 'item1_D'}).drop(columns=['id'])
all_pairs = all_pairs.merge(emobank[['id', 'V']], left_on='item2', right_on='id', how='left').rename(columns={'V': 'item2_V'}).drop(columns=['id'])
all_pairs = all_pairs.merge(emobank[['id', 'A']], left_on='item2', right_on='id', how='left').rename(columns={'A': 'item2_A'}).drop(columns=['id'])
all_pairs = all_pairs.merge(emobank[['id', 'D']], left_on='item2', right_on='id', how='left').rename(columns={'D': 'item2_D'}).drop(columns=['id'])

# Rearrange the columns to have pair_id, item1, item2, item1_split, item2_split, item1_text, item2_text, item1_V, item2_V, item1_A, item2_A, item1_D, item2_D

all_pairs = all_pairs[['item1', 'item2', 'item1_split', 'item2_split', 'item1_text', 'item2_text', 'item1_V', 'item2_V', 'item1_A', 'item2_A', 'item1_D', 'item2_D']]

all_pairs   

In [ ]:
# Check that no item has 3 underscores in its id
assert not all_pairs['item1'].astype(str).str.contains('___').any(), "item1 contains an id with 3 underscores"
assert not all_pairs['item2'].astype(str).str.contains('___').any(), "item2 contains an id with 3 underscores"

In [ ]:
# Create a "pair_id" column that uniquely identifies each pair by concatenating item1 and item2 IDs with 3 underscores as a separator
all_pairs['pair_id'] = all_pairs['item1'].astype(str) + '___' + all_pairs['item2'].astype(str)

# Put pair_id as the first column
cols = ['pair_id'] + [col for col in all_pairs.columns if col != 'pair_id']
all_pairs = all_pairs[cols]

all_pairs

In [ ]:
# all_pairs.to_csv('emopair_pairs.csv', index=False)
# Commented out the above line since we are now loading from the csv instead of generating new pairs each time (seed issues)

## Build Pairadigms

In [ ]:
all_pairs = pd.read_csv('emopair_pairs.csv')
all_pairs

#### Pairwise Annotations

In [ ]:
emopair_aro = pdm.Pairadigm(
    data = all_pairs, 
    paired=True,
    item_id_name='id',
    item_id_cols = ['item1', 'item2'],
    item_text_cols = ['item1_text', 'item2_text'],
    cgcot_prompts=aro_prompts,
    model_name='gpt-5.1', 
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url="https://us.api.openai.com/v1", 
    target_concept='arousal'
)

emopair_dom = pdm.Pairadigm(
    data = all_pairs, 
    paired=True,
    item_id_name='id',
    item_id_cols = ['item1', 'item2'],
    item_text_cols = ['item1_text', 'item2_text'],
    cgcot_prompts=dom_prompts,
    model_name='claude-sonnet-4-5', 
    api_key=os.getenv('ANTHROPIC_API_KEY'),
    target_concept='dominance'
)

#### Arousal

In [ ]:
# emopair_aro.generate_breakdowns_from_paired()

In [ ]:
# emopair_aro.generate_pairwise_annotations()

In [ ]:
# emopair_aro.save("Results/pairadigms/emopair_aro_untied_leakFree")

In [ ]:
emopair_aro = pdm.Pairadigm.load("Results/pairadigms/emopair_aro_untied_leakFree")

In [ ]:
# Check for ERRORs
emopair_aro.pairwise_df['decision'].value_counts()

In [ ]:
aro_examine = emopair_aro.pairwise_df.copy()

aro_examine['higher_A_item'] = 'tie'
aro_examine.loc[aro_examine['item1_A'] > aro_examine['item2_A'], 'higher_A_item'] = 'Text1'
aro_examine.loc[aro_examine['item1_A'] < aro_examine['item2_A'], 'higher_A_item'] = 'Text2'

aro_examine['decision_match_score'] = (aro_examine['higher_A_item'] == aro_examine['decision'])
aro_examine['decision_match_score'].value_counts()

#### Dominance

In [ ]:
# emopair_dom.generate_breakdowns_from_paired()

In [ ]:
# emopair_dom.generate_pairwise_annotations()

In [ ]:
# emopair_dom.save("Results/pairadigms/emopair_dom_untied_leakFree")

In [ ]:
emopair_dom = pdm.Pairadigm.load("Results/pairadigms/emopair_dom_untied_leakFree")

In [ ]:
# Check for ERRORs
emopair_dom.pairwise_df['decision'].value_counts()

In [ ]:
error_rows =emopair_dom.pairwise_df[emopair_dom.pairwise_df['decision'] == 'ERROR'].copy()
error_rows

In [ ]:
dom_examine = emopair_dom.pairwise_df.copy()

dom_examine['higher_A_item'] = 'tie'
dom_examine.loc[dom_examine['item1_A'] > dom_examine['item2_A'], 'higher_A_item'] = 'Text1'
dom_examine.loc[dom_examine['item1_A'] < dom_examine['item2_A'], 'higher_A_item'] = 'Text2'

dom_examine['decision_match_score'] = (dom_examine['higher_A_item'] == dom_examine['decision'])
dom_examine['decision_match_score'].value_counts()

## Score Items

In [ ]:
emopair_aro.score_items(normalization_scale='negative-one-to-one')

In [ ]:
emopair_dom.score_items(normalization_scale='negative-one-to-one')

In [ ]:
import matplotlib.pyplot as plt

# Plot the distribution
plt.figure(figsize=(10, 6))
emopair_aro.scored_df['Bradley_Terry_Score'].hist(bins=20, alpha=0.7, 
                                                  color='skyblue', edgecolor='black')

# Calculate mean and median
mean_bt = emopair_aro.scored_df['Bradley_Terry_Score'].mean()
median_bt = emopair_aro.scored_df['Bradley_Terry_Score'].median()

# Add vertical lines for mean and median
plt.axvline(mean_bt, color='red', linestyle='dashed', 
            linewidth=2, label=f'Mean: {mean_bt:.2f}')
plt.axvline(median_bt, color='green', linestyle='dashed', 
            linewidth=2, label=f'Median: {median_bt:.2f}')

# Add labels and title
plt.title('Distribution of Arousal Bradley-Terry Scores')
plt.xlabel('Bradley-Terry Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
import seaborn as sns
from scipy import stats

plot_data = emopair_aro.scored_df.copy()
plot_data = plot_data.merge(
    emobank[['id', 'A']], 
    left_on='item_id', 
    right_on='id', 
    how='left').drop(columns=['id'])

# Calculate correlation statistics
r, p = stats.spearmanr(plot_data['A'], 
                       plot_data['Bradley_Terry_Score'])

# Create the plot
plt.figure(figsize=(8, 6))
sns.regplot(data=plot_data, x='A', y='Bradley_Terry_Score', 
            scatter_kws={'alpha':0.5}, line_kws={'color': 'red'})

# Add labels and title with stats
plt.title(f'Arousal (A) vs Bradley-Terry Score\nSpearman ρ={r:.2f}, p={p:.2e}')
plt.xlabel('Original Arousal (A)')
plt.ylabel('Bradley-Terry Score')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Plot the distribution
plt.figure(figsize=(10, 6))
emopair_dom.scored_df['Bradley_Terry_Score'].hist(bins=20, alpha=0.7, 
                                                  color='skyblue', edgecolor='black')

# Calculate mean and median
mean_bt = emopair_dom.scored_df['Bradley_Terry_Score'].mean()
median_bt = emopair_dom.scored_df['Bradley_Terry_Score'].median()

# Add vertical lines for mean and median
plt.axvline(mean_bt, color='red', linestyle='dashed', 
            linewidth=2, label=f'Mean: {mean_bt:.2f}')
plt.axvline(median_bt, color='green', linestyle='dashed', 
            linewidth=2, label=f'Median: {median_bt:.2f}')

# Add labels and title
plt.title('Distribution of Dominance Bradley-Terry Scores')
plt.xlabel('Bradley-Terry Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
import seaborn as sns
from scipy import stats

plot_data = emopair_dom.scored_df.copy()
plot_data = plot_data.merge(
    emobank[['id', 'D']], 
    left_on='item_id', 
    right_on='id', 
    how='left').drop(columns=['id'])

# Calculate correlation statistics
r, p = stats.spearmanr(plot_data['D'], 
                       plot_data['Bradley_Terry_Score'])

# Create the plot
plt.figure(figsize=(8, 6))
sns.regplot(data=plot_data, x='D', y='Bradley_Terry_Score', 
            scatter_kws={'alpha':0.5}, line_kws={'color': 'red'})

# Add labels and title with stats
plt.title(f'Dominance (D) vs Bradley-Terry Score\nSpearman ρ={r:.2f}, p={p:.2e}')
plt.xlabel('Original Dominance (D)')
plt.ylabel('Bradley-Terry Score')
plt.grid(True, alpha=0.3)
plt.show()

# Save EmoPair pairadigm data

In [ ]:
date = pd.Timestamp.now().strftime("%Y-%m-%d")
emopair_aro.pairwise_df.to_csv(f'Results/emopair_aro_untied_pairwise_annotations_{date}.csv', index=False)
emopair_aro.scored_df.to_csv(f'Results/emopair_aro_untied_scores_{date}.csv', index=False)

emopair_dom.pairwise_df.to_csv(f'Results/emopair_dom_untied_pairwise_annotations_{date}.csv', index=False)
emopair_dom.scored_df.to_csv(f'Results/emopair_dom_untied_scores_{date}.csv', index=False)

In [ ]:
# emopair_dom.check_transitivity() WILL TAKE A LONG TIME TO RUN, SO COMMENTING OUT FOR NOW.

# Make EmoPair

In [ ]:
cols_to_rename = ['breakdown1', 'breakdown2', 'decision', 'justification']
dimension = ['A', 'D']

In [ ]:
aro_scores = emopair_aro.scored_df.rename(columns={'Bradley_Terry_Score': 'A_BT_score_full'}).copy()
dom_scores = emopair_dom.scored_df.rename(columns={'Bradley_Terry_Score': 'D_BT_score_full'}).copy()

In [ ]:
# Filter pairs by split combinations
aro_train_pairs = emopair_aro.pairwise_df[
    (emopair_aro.pairwise_df['item1_split'] == 'train') & 
    (emopair_aro.pairwise_df['item2_split'] == 'train')
].copy()

aro_dev_pairs = emopair_aro.pairwise_df[
    (emopair_aro.pairwise_df['item1_split'] == 'dev') & 
    (emopair_aro.pairwise_df['item2_split'] == 'dev')
].copy()

aro_test_pairs = emopair_aro.pairwise_df[
    (emopair_aro.pairwise_df['item1_split'] == 'test') & 
    (emopair_aro.pairwise_df['item2_split'] == 'test')
].copy()

dom_train_pairs = emopair_dom.pairwise_df[
    (emopair_dom.pairwise_df['item1_split'] == 'train') & 
    (emopair_dom.pairwise_df['item2_split'] == 'train')
].copy()

dom_dev_pairs = emopair_dom.pairwise_df[
    (emopair_dom.pairwise_df['item1_split'] == 'dev') & 
    (emopair_dom.pairwise_df['item2_split'] == 'dev')
].copy()

dom_test_pairs = emopair_dom.pairwise_df[
    (emopair_dom.pairwise_df['item1_split'] == 'test') & 
    (emopair_dom.pairwise_df['item2_split'] == 'test')
].copy()

print(f"Arousal train/train pairs: {len(aro_train_pairs)}")
print(f"Arousal dev/dev pairs: {len(aro_dev_pairs)}")
print(f"Arousal test/test pairs: {len(aro_test_pairs)}")
print(f"Dominance train/train pairs: {len(dom_train_pairs)}")
print(f"Dominance dev/dev pairs: {len(dom_dev_pairs)}")
print(f"Dominance test/test pairs: {len(dom_test_pairs)}")

# Create temporary Pairadigm objects for each split
splits = ['train', 'dev', 'test']
aro_split_pairadigms = {}
dom_split_pairadigms = {}

for split in splits:
    # Arousal
    aro_split_pairadigms[split] = pdm.Pairadigm(
        data=all_pairs,
        paired=True,
        item_id_name='id',
        item_id_cols=['item1', 'item2'],
        item_text_cols=['item1_text', 'item2_text'],
        cgcot_prompts=aro_prompts,
        model_name='gpt-5.1',
        api_key=os.getenv('OPENAI_API_KEY'),
        base_url="https://us.api.openai.com/v1",
        target_concept='arousal'
    )
    
    # Dominance
    dom_split_pairadigms[split] = pdm.Pairadigm(
        data=all_pairs,
        paired=True,
        item_id_name='id',
        item_id_cols=['item1', 'item2'],
        item_text_cols=['item1_text', 'item2_text'],
        cgcot_prompts=dom_prompts,
        model_name='claude-sonnet-4-5',
        api_key=os.getenv('ANTHROPIC_API_KEY'),
        target_concept='dominance'
    )

# Assign filtered pairs to corresponding Pairadigm objects
aro_split_pairadigms['train'].pairwise_df = aro_train_pairs
aro_split_pairadigms['dev'].pairwise_df = aro_dev_pairs
aro_split_pairadigms['test'].pairwise_df = aro_test_pairs

dom_split_pairadigms['train'].pairwise_df = dom_train_pairs
dom_split_pairadigms['dev'].pairwise_df = dom_dev_pairs
dom_split_pairadigms['test'].pairwise_df = dom_test_pairs

# Score items for each split and combine into single dataframes
split_scores_list = []

for split in splits:
    # Score arousal
    aro_split_pairadigms[split].score_items(normalization_scale='negative-one-to-one')
    aro_split_scores = aro_split_pairadigms[split].scored_df[['item_id', 'Bradley_Terry_Score']].rename(
        columns={'Bradley_Terry_Score': 'A_BT_score_split'}
    )
    
    # Score dominance
    dom_split_pairadigms[split].score_items(normalization_scale='negative-one-to-one')
    dom_split_scores = dom_split_pairadigms[split].scored_df[['item_id', 'Bradley_Terry_Score']].rename(
        columns={'Bradley_Terry_Score': 'D_BT_score_split'}
    )
    
    # Merge arousal and dominance scores for this split
    split_score = aro_split_scores.merge(dom_split_scores, on='item_id', how='outer')
    split_scores_list.append(split_score)
    
    print(f"\n{split.capitalize()} scores:")
    print(split_score.head())

split_scores = pd.concat(split_scores_list, ignore_index=True)

In [ ]:
split_scores

In [ ]:
# Combine the A and D scores into a single dataframe
all_scores = pd.merge(aro_scores, dom_scores, on='item_id', how='inner')

# Merge the original A and D ratings from emobank into all_scores
all_scores = emobank.merge(
    all_scores,
    left_on='id',
    right_on='item_id',
    how='left').drop(columns=['id'])

# Rescale the original A and D ratings from emobank to be between -1 and 1 instead of 1 and 5
all_scores['A_rescaled'] = (all_scores['A'] - 3) / 2
all_scores['D_rescaled'] = (all_scores['D'] - 3) / 2

# Merge the split scores into all_scores
all_scores = all_scores.merge(split_scores, on='item_id', how='left')

# Put item_id as the first column
cols = ['item_id'] + [col for col in all_scores.columns if col != 'item_id']
all_scores = all_scores[cols]

all_scores

In [ ]:
all_scores.to_csv('Results/emopair.csv', index=False)

In [ ]:
aro_pair_breakdowns = emopair_aro.pairwise_df.copy().rename(columns={
    'breakdown1': 'A_breakdown1',
    'breakdown2': 'A_breakdown2',
    'decision': 'A_decision',
    'justification': 'A_justification'
})

dom_pair_breakdowns = emopair_dom.pairwise_df.copy().rename(columns={
    'breakdown1': 'D_breakdown1',
    'breakdown2': 'D_breakdown2',
    'decision': 'D_decision',
    'justification': 'D_justification'
})[['pair_id', 'D_breakdown1', 'D_breakdown2', 'D_decision', 'D_justification']]

all_breakdowns = aro_pair_breakdowns.merge(
    dom_pair_breakdowns,
    on='pair_id')

all_breakdowns

In [ ]:
all_breakdowns.to_csv('emopair_pairs_w_breakdowns.csv', index=False)

# EVERYTHING PAST THIS IS OUT OF DATE, DATA LEAKAGE

Kept pieces related to the tie phenomenon we identified before in case we need to revisit. 

## Build pairadigms

In [ ]:
# emopair_aro.score_items(normalization_scale='negative-one-to-one', 
#                         use_davidson=True)
# run time: 85 minutes

In [ ]:
# emopair_aro.scored_df

# import matplotlib.pyplot as plt

# # Plot the distribution
# plt.figure(figsize=(10, 6))
# emopair_aro.scored_df['Davidson_Score'].hist(bins=30, alpha=0.7, color='skyblue', edgecolor='black')

# # Calculate mean and median
# mean_score = emopair_aro.scored_df['Davidson_Score'].mean()
# median_score = emopair_aro.scored_df['Davidson_Score'].median()

# # Add vertical lines for mean and median
# plt.axvline(mean_score, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_score:.2f}')
# plt.axvline(median_score, color='green', linestyle='dashed', linewidth=2, label=f'Median: {median_score:.2f}')

# # Add labels and title
# plt.title('Distribution of Davidson Scores')
# plt.xlabel('Davidson Score')
# plt.ylabel('Frequency')
# plt.legend()
# plt.show()

In [ ]:
emopair_aro.pairwise_df['decision'].value_counts()

In [ ]:
# import seaborn as sns
# from scipy import stats

# # Calculate correlation statistics
# r, p = stats.spearmanr(emopair_aro.scored_df['A'], emopair_aro.scored_df['Davidson_Score'])

# # Create the plot
# plt.figure(figsize=(8, 6))
# sns.regplot(data=emopair_aro.scored_df, x='A', y='Davidson_Score', 
#             scatter_kws={'alpha':0.5}, line_kws={'color': 'red'})

# # Add labels and title with stats
# plt.title(f'Arousal (A) vs Davidson Score\nSpearman ρ={r:.2f}, p={p:.2e}')
# plt.xlabel('Original Arousal (A)')
# plt.ylabel('Davidson Score')
# plt.grid(True, alpha=0.3)
# plt.show()

In [ ]:
# Rescale 'A' from original (likely 1-5) to -1 to 1
# A_min, A_max = emopair_aro.scored_df['A'].min(), emopair_aro.scored_df['A'].max()
# emopair_aro.scored_df['A_rescaled'] = 2 * (emopair_aro.scored_df['A'] - A_min) / (A_max - A_min) - 1

plt.figure(figsize=(10, 6))

# Plot histogram for 'A'
plt.hist(emopair_aro.scored_df['A_rescaled'], bins=30, alpha=0.6, color='orange', label='Original Arousal (A)', edgecolor='black')

# Plot histogram for 'Davidson_Score'
plt.hist(emopair_aro.scored_df['Davidson_Score'], bins=30, alpha=0.6, color='skyblue', label='Davidson Score', edgecolor='black')

plt.title('Distribution of Original Arousal (A) and Davidson Score')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## Untied Version

After interrogating the highly skewed distribution and high prevalence of ties in the pairwise annotations, we will build an untied version of the pairadigms by not allowing ties during the pairwise annotation generation step.

From our work in `validate_LLMs.ipynb`, we found that the following LLMs were the best performers without ties allowed:
- Valence: N/A
- Arousal: GPT-5.1
- Dominance: Claude-Sonnet 4.5

Note, everything in the Prep section above needs to be run first.

In [ ]:
# emopair_aro_untied.score_items(normalization_scale='negative-one-to-one')

In [ ]:
import matplotlib.pyplot as plt

# Plot the distribution
plt.figure(figsize=(10, 6))
emopair_aro_untied.scored_df['Bradley_Terry_Score'].hist(bins=20, alpha=0.7, color='skyblue', edgecolor='black')

# Calculate mean and median
mean_bt = emopair_aro_untied.scored_df['Bradley_Terry_Score'].mean()
median_bt = emopair_aro_untied.scored_df['Bradley_Terry_Score'].median()

# Add vertical lines for mean and median
plt.axvline(mean_bt, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_bt:.2f}')
plt.axvline(median_bt, color='green', linestyle='dashed', linewidth=2, label=f'Median: {median_bt:.2f}')

# Add labels and title
plt.title('Distribution of Bradley-Terry Scores')
plt.xlabel('Bradley-Terry Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
import seaborn as sns
from scipy import stats

# Calculate correlation statistics
r, p = stats.spearmanr(emopair_aro_untied.scored_df['A'], emopair_aro_untied.scored_df['Bradley_Terry_Score'])

# Create the plot
plt.figure(figsize=(8, 6))
sns.regplot(data=emopair_aro_untied.scored_df, x='A', y='Bradley_Terry_Score', 
            scatter_kws={'alpha':0.5}, line_kws={'color': 'red'})

# Add labels and title with stats
plt.title(f'Arousal (A) vs Bradley-Terry Score\nSpearman ρ={r:.2f}, p={p:.2e}')
plt.xlabel('Original Arousal (A)')
plt.ylabel('Davidson Score')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Rescale 'A' from original (likely 1-5) to -1 to 1
A_min, A_max = emopair_aro_untied.scored_df['A'].min(), emopair_aro_untied.scored_df['A'].max()
emopair_aro_untied.scored_df['A_rescaled'] = 2 * (emopair_aro_untied.scored_df['A'] - A_min) / (A_max - A_min) - 1

plt.figure(figsize=(10, 6))

# Plot histogram for 'A'
plt.hist(emopair_aro_untied.scored_df['A_rescaled'], bins=30, alpha=0.6, color='orange', label='Original Arousal (A)', edgecolor='black')

# Plot histogram for 'Davidson_Score'
plt.hist(emopair_aro_untied.scored_df['Bradley_Terry_Score'], bins=30, alpha=0.6, color='skyblue', label='Bradley-Terry Score', edgecolor='black')

plt.title('Distribution of Original Arousal (A) and Bradley-Terry Score')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## Dominance

In [ ]:
mask = emopair_dom_untied.pairwise_df['decision'] == 'ERROR'
emopair_dom_untied.pairwise_df[mask]

In [ ]:
# emopair_dom_untied.score_items(normalization_scale='negative-one-to-one')

In [ ]:
import matplotlib.pyplot as plt

# Plot the distribution
plt.figure(figsize=(10, 6))
emopair_dom_untied.scored_df['Bradley_Terry_Score'].hist(bins=20, alpha=0.7, color='skyblue', edgecolor='black')

# Calculate mean and median
mean_bt = emopair_dom_untied.scored_df['Bradley_Terry_Score'].mean()
median_bt = emopair_dom_untied.scored_df['Bradley_Terry_Score'].median()
# Add vertical lines for mean and median
plt.axvline(mean_bt, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_bt:.2f}')
plt.axvline(median_bt, color='green', linestyle='dashed', linewidth=2, label=f'Median: {median_bt:.2f}')

# Add labels and title
plt.title('Distribution of Bradley-Terry Scores')
plt.xlabel('Bradley-Terry Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
import seaborn as sns
from scipy import stats

# Calculate correlation statistics
r, p = stats.spearmanr(emopair_dom_untied.scored_df['D'], emopair_dom_untied.scored_df['Bradley_Terry_Score'])

# Create the plot
plt.figure(figsize=(8, 6))
sns.regplot(data=emopair_dom_untied.scored_df, x='D', y='Bradley_Terry_Score', 
            scatter_kws={'alpha':0.5}, line_kws={'color': 'red'})

# Add labels and title with stats
plt.title(f'Dominance (D) vs Bradley-Terry Score\nSpearman ρ={r:.2f}, p={p:.2e}')
plt.xlabel('Original Dominance (D)')
plt.ylabel('Score')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Rescale 'A' from original (likely 1-5) to -1 to 1
D_min, D_max = emopair_dom_untied.scored_df['D'].min(), emopair_dom_untied.scored_df['D'].max()
emopair_dom_untied.scored_df['D_rescaled'] = 2 * (emopair_dom_untied.scored_df['D'] - D_min) / (D_max - D_min) - 1

plt.figure(figsize=(10, 6))

# Plot histogram for 'A'
plt.hist(emopair_dom_untied.scored_df['D_rescaled'], bins=30, alpha=0.6, color='orange', label='Original Dominance (D)', edgecolor='black')

# Plot histogram for 'Davidson_Score'
plt.hist(emopair_dom_untied.scored_df['Bradley_Terry_Score'], bins=30, alpha=0.6, color='skyblue', label='Bradley-Terry Score', edgecolor='black')

plt.title('Distribution of Original Dominance (D) and Bradley-Terry Score')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()